# Final locked Sketch evaluation
Written only; no cells executed during creation. See README.md for run order.

In [ ]:
from pathlib import Path
import json
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if p.name == 'Task 3'
             and (p / 'task3/configs/erm.yaml').is_file()), None)
if REPO is None:
    raise RuntimeError('Open Jupyter inside Task 3.')
ROOT = REPO.parent
TASK2 = ROOT / 'Task 2'
DATA_ROOT = ROOT / 'data/PACS'
SPLIT_PATH = TASK2 / 'shared/splits/pacs_sketch_seed6304.json'
BASELINE = TASK2 / 'task2/results/source_only/best.pt'
RESULTS = REPO / 'task3/results'
CONFIG = json.loads((REPO / 'task3/configs/erm.yaml').read_text())

def load_notebook(path):
    path = Path(path)
    document = json.loads(path.read_text(encoding='utf-8'))
    for index, cell in enumerate(document['cells']):
        if cell['cell_type'] == 'code':
            source = cell['source']
            source = ''.join(source) if isinstance(source, list) else source
            exec(compile(source, f'{path}:cell-{index}', 'exec'), globals())

# Only definitions are loaded here; no target dataset is constructed.
load_notebook(TASK2 / 'shared/pacs_protocol.ipynb')
load_notebook(TASK2 / 'shared/pacs.ipynb')
load_notebook(REPO / 'task3/models/classifier_head.ipynb')
load_notebook(REPO / 'task3/models/backbone.ipynb')
load_notebook(TASK2 / 'task2/methods/dan.ipynb')  # Exact shared MMD implementation.

import random
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

load_notebook(REPO / 'task3/selection/source_validation.ipynb')
load_notebook(REPO / 'task3/methods/erm.ipynb')
load_notebook(REPO / 'task3/methods/dan_dg.ipynb')
load_notebook(REPO / 'task3/methods/sam.ipynb')
load_notebook(REPO / 'task3/evaluation/source_domain_separability.ipynb')
load_notebook(REPO / 'task3/evaluation/sharpness.ipynb')
load_notebook(REPO / 'task3/evaluation/domain_metrics.ipynb')


In [ ]:
"""Task 2 final analysis only: target labels must not feed back into training."""
import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler



def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


class EvaluationImages(Dataset):
    def __init__(self, root, records):
        self.root, self.records = Path(root), records
        self.transform = image_transform(False)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        r = self.records[index]
        with Image.open(self.root / r['path']) as im:
            return self.transform(im.convert('RGB')), r['label']


@torch.no_grad()
def extract(model, root, records, device):
    loader = DataLoader(EvaluationImages(root, records), batch_size=64, shuffle=False, num_workers=0)
    features, logits = [], []
    for images, _ in loader:
        f = forward_features(model, images.to(device))
        z = model.fc(f)
        if not torch.isfinite(f).all() or not torch.isfinite(z).all():
            raise RuntimeError('Non-finite evaluation features/logits; inspect the training run.')
        features.append(f.cpu().numpy()); logits.append(z.cpu().numpy())
    return np.concatenate(features), np.concatenate(logits)




In [ ]:
"""Final Task 3 target evaluation. Run only after the study and source diagnostics."""
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix



def evaluate_sketch(root):
    root = Path(root)
    lock = freeze_protocol(root)  # All five completed checkpoints/configurations fixed first.
    out = root / 'Task 3/task3/results/final_evaluation'
    summary = pd.read_csv(out / 'source_diagnostics.csv')
    if set(summary.method) != set(runs(root)):
        raise ValueError('Complete all source diagnostics before opening Sketch.')
    data = root / 'data/PACS'
    target, rejected = [], []
    folder = data / 'sketch'
    if sorted(p.name for p in folder.iterdir() if p.is_dir() and not p.name.startswith('.')) != list(CLASSES):
        raise ValueError('Unexpected Sketch classes.')
    for label, name in enumerate(CLASSES):
        for path in sorted((folder / name).rglob('*')):
            if not path.is_file() or path.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp'}:
                continue
            try:
                with Image.open(path) as im:
                    im.convert('RGB').load()
            except (OSError, ValueError) as exc:
                rejected.append(dict(path=path.relative_to(data).as_posix(), reason=str(exc)))
                continue
            target.append(dict(path=path.relative_to(data).as_posix(), label=label, domain='sketch'))
    if set(r['label'] for r in target) != set(range(7)):
        raise ValueError('Missing target classes.')
    write_lock(out / 'target_inventory.json', [dict(r, sha256=file_hash(data / r['path'])) for r in target])
    write_lock(out / 'rejected_target_images.json', rejected)
    print(f'Final Sketch evaluation: {len(target)} images, {len(rejected)} decoding failures.',flush=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    truth = np.array([r['label'] for r in target])
    predictions, classes, confusions, scores = [], [], [], []
    for name, run in runs(root).items():
        print(f'Sketch: {name}',flush=True)
        model = load_model(run / 'best.pt',device)
        _,logits = extract(model,data,target,device)
        pred=logits.argmax(1)
        scores.append(dict(method=name,sketch_accuracy=accuracy_score(truth,pred),
                           sketch_macro_f1=f1_score(truth,pred,labels=range(7),average='macro',zero_division=0)))
        cm=confusion_matrix(truth,pred,labels=range(7))
        for k,c in enumerate(CLASSES):
            wrong=cm[k].copy(); wrong[k]=0
            classes.append(dict(method=name,class_name=c,count=int(cm[k].sum()),accuracy=cm[k,k]/cm[k].sum(),
                                dominant_confusion=CLASSES[wrong.argmax()] if wrong.sum() else '',
                                dominant_confusion_count=int(wrong.max())))
            for j,pc in enumerate(CLASSES):
                confusions.append(dict(method=name,true_class=c,predicted_class=pc,count=int(cm[k,j])))
        predictions.extend(dict(method=name,path=r['path'],label=r['label'],predicted_label=int(p),
                                correct=bool(p==r['label'])) for r,p in zip(target,pred))
        del model
    summary=summary.merge(pd.DataFrame(scores),on='method',validate='one_to_one')
    baseline=summary.loc[summary.method=='ERM','sketch_accuracy'].iloc[0]
    summary['sketch_accuracy_change_pp']=100*(summary.sketch_accuracy-baseline)
    per_class=pd.DataFrame(classes)
    base=per_class[per_class.method=='ERM'].set_index('class_name').accuracy
    per_class['accuracy_change_pp']=100*(per_class.accuracy-per_class.class_name.map(base))
    predictions=pd.DataFrame(predictions)
    base_pred=predictions[predictions.method=='ERM'].set_index('path')
    predictions['baseline_correct']=predictions.path.map(base_pred.correct)
    predictions['baseline_prediction']=predictions.path.map(base_pred.predicted_label)
    cases=predictions[predictions.correct!=predictions.baseline_correct].copy()
    cases['class_name']=cases.label.map(dict(enumerate(CLASSES)))
    cases['change']=np.where(cases.correct,'corrected','introduced_error')
    cases=cases.groupby(['method','class_name','change'],sort=False).head(3)
    study=summary[summary.method.str.startswith('DAN-DG')].copy()
    study.insert(1,'lambda_dg',study.method.map({'DAN-DG':1.,'DAN-DG lambda=0.1':0.1,'DAN-DG lambda=10':10.}))
    study=study.sort_values('lambda_dg')
    for filename,table in [('comparison.csv',summary),('main_comparison.csv',summary[summary.method.isin(['ERM','DAN-DG','SAM'])]),
                           ('per_class.csv',per_class),('target_predictions.csv',predictions),
                           ('confusions.csv',pd.DataFrame(confusions)),('selected_cases.csv',cases),('controlled_study.csv',study)]:
        table.to_csv(out / filename,index=False)
    # Read Task 2 target results only after every Task 3 decision and evaluation is fixed.
    t2=root / 'Task 2/task2/results/final_evaluation'
    if (t2 / 'comparison.csv').exists():
        previous=pd.read_csv(t2 / 'comparison.csv')
        previous=previous[previous.method.isin(['Source-only','DAN'])].copy()
        previous.to_csv(out / 'task2_reference.csv',index=False)
        prior_classes=pd.read_csv(t2 / 'per_class.csv')
        prior_classes[prior_classes.method.isin(['Source-only','DAN'])].to_csv(out / 'task2_per_class_reference.csv',index=False)
        old_inventory=json.loads((t2 / 'image_manifest.json').read_text())
        old_targets={r['path'] for r in old_inventory if r['domain']=='sketch'}
        old_lock=json.loads((t2 / 'evaluation_lock.json').read_text())
        check=dict(same_target_paths=old_targets=={r['path'] for r in target},
                   same_erm_checkpoint=old_lock['runs']['Source-only']['checkpoint_sha256']==lock['runs']['ERM']['sha256'],
                   same_split=old_lock['split_sha256']==lock['split_sha256'])
        (out / 'task2_comparison_verification.json').write_text(json.dumps(check,indent=2))
        if not all(check.values()):
            raise ValueError('Task 2 reference comparability checks failed.')
    return summary,per_class,cases


In [ ]:
"""Plots and evidence tables for the completed Task 3 experiments."""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image



def make_figures(root):
    root=Path(root); out=root / 'Task 3/task3/results/final_evaluation'
    summary=pd.read_csv(out/'comparison.csv')
    main=summary[summary.method.isin(['ERM','DAN-DG','SAM'])]
    study=pd.read_csv(out/'controlled_study.csv')
    classes=pd.read_csv(out/'per_class.csv')
    cases=pd.read_csv(out/'selected_cases.csv')
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    x=np.arange(len(main)); width=.25
    for i,(column,label) in enumerate([('mean_source_accuracy','Mean source'),('worst_source_accuracy','Worst source'),('sketch_accuracy','Sketch')]):
        axes[0].bar(x+(i-1)*width,100*main[column],width,label=label)
    axes[0].set(xticks=x,xticklabels=main.method,ylabel='Accuracy (%)',ylim=(0,105)); axes[0].legend(fontsize=8)
    axes[1].bar(main.method,100*main.source_domain_separability)
    axes[1].axhline(100/3,color='black',ls='--',label='Chance (33.3%)')
    axes[1].set(ylabel='Source domain probe accuracy (%)',ylim=(0,105)); axes[1].legend(fontsize=8)
    axes[2].bar(main.method,main.sharpness); axes[2].set(ylabel='Validation CE increase',title='Common sharpness proxy (radius 0.05)')
    fig.tight_layout(); fig.savefig(out/'main_comparison.png',dpi=160); plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for column,label in [('mean_source_accuracy','Mean source'),('worst_source_accuracy','Worst source'),('sketch_accuracy','Sketch')]:
        axes[0].plot(study.lambda_dg,100*study[column],marker='o',label=label)
    axes[0].set(xscale='log',xlabel='DAN-DG lambda',ylabel='Accuracy (%)'); axes[0].legend()
    axes[1].plot(study.lambda_dg,100*study.source_domain_separability,marker='o')
    axes[1].axhline(100/3,color='black',ls='--'); axes[1].set(xscale='log',xlabel='DAN-DG lambda',ylabel='Source domain probe accuracy (%)')
    fig.tight_layout(); fig.savefig(out/'controlled_study.png',dpi=160); plt.close(fig)
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for ax,(name,folder) in zip(axes.flat,runs(root).items()):
        h=pd.read_csv(folder/'history.csv')
        column='classification_loss' if 'classification_loss' in h else 'train_loss'
        ax.plot(h.epoch,h[column],label='Classification CE')
        if 'mmd_loss' in h:
            ax.plot(h.epoch,h.mmd_loss,label='Mean pairwise MMD')
            ax.plot(h.epoch,h.train_loss,label='Total objective')
        if 'perturbed_classification_loss' in h:
            ax.plot(h.epoch,h.perturbed_classification_loss,label='Perturbed CE')
        ax.set(title=name,xlabel='Epoch',ylabel='Loss'); ax.legend(fontsize=8)
    axes.flat[-1].axis('off'); fig.tight_layout(); fig.savefig(out/'all_training_curves.png',dpi=160); plt.close(fig)
    fig,ax=plt.subplots(figsize=(10,4)); x=np.arange(7)
    for i,name in enumerate(['DAN-DG','SAM']):
        values=classes[classes.method==name].set_index('class_name').loc[list(CLASSES)]
        ax.bar(x+(i-.5)*.35,values.accuracy_change_pp,.35,label=name)
    ax.axhline(0,color='black',lw=.8); ax.set(xticks=x,xticklabels=CLASSES,ylabel='Sketch accuracy change vs ERM (pp)'); ax.legend()
    fig.tight_layout(); fig.savefig(out/'per_class_changes.png',dpi=160); plt.close(fig)
    conf=pd.read_csv(out/'confusions.csv')
    fig,axes=plt.subplots(1,3,figsize=(16,5))
    for ax,name in zip(axes,['ERM','DAN-DG','SAM']):
        cm=conf[conf.method==name].pivot(index='true_class',columns='predicted_class',values='count').loc[list(CLASSES),list(CLASSES)].to_numpy()
        normalized=cm/cm.sum(1,keepdims=True)
        ax.imshow(normalized,vmin=0,vmax=1,cmap='Blues')
        for i in range(7):
            for j in range(7): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=7,color='white' if normalized[i,j]>.5 else 'black')
        ax.set(title=name,xticks=range(7),xticklabels=CLASSES,yticks=range(7),yticklabels=CLASSES,xlabel='Predicted',ylabel='True')
        ax.tick_params(axis='x',rotation=60)
    fig.tight_layout(); fig.savefig(out/'sketch_confusions.png',dpi=160); plt.close(fig)
    # Choose deterministic examples from classes with largest improvement/degradation.
    selected=[]
    for name in ['DAN-DG','SAM']:
        values=classes[classes.method==name]
        for change,ascending in [('corrected',False),('introduced_error',True)]:
            ranked=values.sort_values('accuracy_change_pp',ascending=ascending).class_name
            for c in ranked:
                subset=cases[(cases.method==name)&(cases.change==change)&(cases.class_name==c)]
                if len(subset):
                    selected.extend(subset.head(2).to_dict('records')); break
    if selected:
        cols=4; rows=(len(selected)+cols-1)//cols
        fig,axes=plt.subplots(rows,cols,figsize=(14,3.8*rows),squeeze=False)
        for ax in axes.flat: ax.axis('off')
        for ax,r in zip(axes.flat,selected):
            with Image.open(root/'data/PACS'/r['path']) as im: ax.imshow(im.convert('RGB'))
            ax.set_title(f"{r['method']}: {r['change']}\nTrue: {r['class_name']} | ERM: {CLASSES[int(r['baseline_prediction'])]}\nNew: {CLASSES[int(r['predicted_label'])]}",fontsize=9)
        fig.tight_layout(); fig.savefig(out/'selected_failures.png',dpi=150); plt.close(fig)
        pd.DataFrame(selected).to_csv(out/'illustrated_cases.csv',index=False)
    if not (out/'task2_reference.csv').exists():
        print('Task 2 final comparison is pending. Run its final evaluation, then rerun this notebook for cross-task tables.')
        return main, study, pd.DataFrame()
    previous=pd.read_csv(out/'task2_reference.csv')
    t2classes=pd.read_csv(out/'task2_per_class_reference.csv')
    dan=previous[previous.method=='DAN'].iloc[0]
    baseline=main[main.method=='ERM'].iloc[0]
    cross=pd.DataFrame([dict(method='Task 2 DAN (target-aware)',sketch_accuracy=dan.target_accuracy,
                             sketch_macro_f1=dan.target_macro_f1,sketch_accuracy_change_pp=100*(dan.target_accuracy-baseline.sketch_accuracy)),
                        dict(method='Task 3 DAN-DG (source-only)',**main[main.method=='DAN-DG'][['sketch_accuracy','sketch_macro_f1','sketch_accuracy_change_pp']].iloc[0].to_dict())])
    cross.to_csv(out/'dan_vs_dan_dg.csv',index=False)
    t2=t2classes[t2classes.method=='DAN'][['class_name','accuracy']].rename(columns={'accuracy':'task2_dan_accuracy'})
    t3=classes[classes.method=='DAN-DG'][['class_name','accuracy']].rename(columns={'accuracy':'task3_dan_dg_accuracy'})
    t2.merge(t3,on='class_name').to_csv(out/'dan_vs_dan_dg_per_class.csv',index=False)
    return main,study,cross


In [ ]:
ALL_DECISIONS_LOCKED = False
if not ALL_DECISIONS_LOCKED:
    raise RuntimeError('Complete all training and source diagnostics; freeze decisions before enabling Sketch evaluation.')
summary, per_class, cases = evaluate_sketch(ROOT)
display(summary)
display(per_class)
main, study, cross_task = make_figures(ROOT)
display(study)
display(cross_task)
